# g7 Inference Price-Performance — Nemotron-Nano-30B-A3B-NVFP4

Compares **cost-per-token and latency across g7 (RTX PRO 4500 Blackwell) sizes** for
`NVIDIA-Nemotron-3-Nano-30B-A3B-NVFP4`, using the **SageMaker AI Inference Recommender** SDK
(`ModelBuilder` → `generate_deployment_recommendations` → `mb.recommendations`).

**Prerequisites**
- An AWS account with SageMaker access and **g7 available in your Region**.
- An IAM execution role (set as `SAGEMAKER_ROLE_ARN`) with `AmazonSageMakerFullAccess`, S3 read on
  your model bucket, and `servicequotas:GetServiceQuota` / `ListServiceQuotas`.
- Your model artifacts staged in S3 (set `MODEL_S3` below). This uses an **S3 model source**; a
  JumpStart id would require g7 in that model's inference-container config.


## Setup

Install the SageMaker SDK, then resolve the execution role + Region from your environment. The AI
Inference Recommender ships in the SageMaker SDK — install the released packages that provide
`sagemaker.serve.ai_inference_recommender` (see the launch docs for exact package versions).

In [ ]:
%pip install -q sagemaker-core sagemaker-serve sagemaker-train

In [ ]:
import os
from sagemaker.core.helper.session_helper import Session, get_execution_role

# Set SAGEMAKER_ROLE_ARN (works anywhere), or run inside SageMaker Studio to auto-resolve it.
ROLE = os.environ.get("SAGEMAKER_ROLE_ARN") or get_execution_role(sagemaker_session=Session())
print("execution role:", ROLE)

import logging, sys
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                    stream=sys.stdout, force=True)
log = logging.getLogger("g7-pp")

In [ ]:
# The recommender calls servicequotas:GetServiceQuota + ListServiceQuotas. If your execution role
# doesn't already grant them, attach an inline policy (idempotent; needs iam:PutRolePolicy).
import json
import boto3

_ROLE_NAME = ROLE.split("/")[-1]
_POLICY = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow",
           "Action": ["servicequotas:GetServiceQuota", "servicequotas:ListServiceQuotas"],
           "Resource": "*"}]}
try:
    boto3.client("iam").put_role_policy(
        RoleName=_ROLE_NAME, PolicyName="SageMakerInferenceRecommenderServiceQuotas",
        PolicyDocument=json.dumps(_POLICY))
    print(f"Attached servicequotas inline policy to {_ROLE_NAME!r}.")
except Exception as e:
    print(f"Could not auto-patch {_ROLE_NAME!r}: {e}\nAttach this manually and re-run:\n{json.dumps(_POLICY, indent=2)}")

## Config

In [ ]:
# Your model artifacts in S3 (this notebook uses an S3 model source).
MODEL_S3     = "s3://<YOUR_BUCKET>/models/NVIDIA-Nemotron-3-Nano-30B-A3B-NVFP4/"
# g7 sizes to compare in one job (max 3).
G7_INSTANCES = ["ml.g7.2xlarge", "ml.g7.12xlarge", "ml.g7.48xlarge"]

# Published SageMaker Hosting on-demand $/hr per instance -- fill from
# https://aws.amazon.com/sagemaker/pricing/ for your Region.
PRICE_PER_HR = {
    "ml.g7.2xlarge": None, "ml.g7.4xlarge": None, "ml.g7.8xlarge": None,
    "ml.g7.12xlarge": None, "ml.g7.24xlarge": None, "ml.g7.48xlarge": None,
}

## 1. Recommend across the g7 sizes

One job, up to 3 instances; returns a recommendation per instance.

In [ ]:
from sagemaker.serve import ModelBuilder, InferenceFramework, PerformanceTarget

mb = ModelBuilder(s3_model_data_url=MODEL_S3, role_arn=ROLE)
rec = mb.generate_deployment_recommendations(
    tokenizer="gpt2",
    concurrency=1, request_count=10,
    prompt_input_tokens_mean=512, output_tokens_mean=256,
    performance_target=PerformanceTarget.THROUGHPUT,
    instance_types=G7_INSTANCES,
    advanced_optimization=False,
    framework=InferenceFramework.VLLM,     # NVFP4 serves on vLLM
    wait=True,
)
print("status:", rec.ai_recommendation_job_status)
print(mb.recommendations)                  # per-instance comparison (throughput / latency)

## 2. Price-performance and latency per instance

`$/1M = $/hr x 1e6 / (output tok/s x 3600)` -- lower is better. Latency (time-to-first-token and
inter-token latency) is returned on the same recommendation object, so throughput, cost, and
responsiveness all come from one job. Fill `PRICE_PER_HR` first.

In [ ]:
def _instance(r):
    try: return r.deployment_configuration.instance_type
    except Exception: return r.recommendation_spec_name or "?"

def _stat(metric, stat):
    return getattr(metric, stat, None) if metric is not None else None

print(f"{'instance':16} {'tok/s':>8} {'$/1M':>8} {'TTFT p50':>10} {'ITL p50':>9}")
rows = []
for r in mb.recommendations:
    inst = _instance(r)
    ep = r.expected_performance
    tok  = _stat(ep.output_token_throughput, "avg")
    ttft = _stat(ep.time_to_first_token, "p50")   # ms
    itl  = _stat(ep.inter_token_latency, "p50")    # ms
    hr = PRICE_PER_HR.get(inst)
    permil = round(hr * 1e6 / (float(tok) * 3600), 2) if (hr and tok) else None
    rows.append((inst, tok, permil, ttft, itl))
    print(f"{inst:16} {tok or 0:>8.0f} {('$' + str(permil)) if permil else '-':>8}"
          f" {(f'{ttft:.0f} ms') if ttft else '-':>10} {(f'{itl:.1f} ms') if itl else '-':>9}")

best = min((x for x in rows if x[2] is not None), key=lambda x: x[2], default=None)
if best:
    print(f"\nCheapest per token: {best[0]} at ${best[2]}/1M")

## 3. Full per-concurrency curve (matched-concurrency comparison)

`mb.recommendations` (and `DescribeAIRecommendationJob`) return only each instance's
**service-selected** concurrency. The job also writes the raw AIPerf artifacts for **every**
benchmarked concurrency to its output S3 (`rec.output_config.s3_output_location`). Reading those
gives the full throughput / latency-vs-concurrency curve, and lets you compare instances at a
**matched** concurrency instead of each one's self-selected point.

This reuses the SDK's own profile parser (`BenchmarkMetrics.from_profile_json`). The key layout
`.../benchmark/<node>/.../c<N>/output/output.tar.gz` is the observed recommendation-job output
layout; the cell degrades gracefully if nothing is found (e.g. the job is still running).

In [ ]:
import io, re, tarfile
from sagemaker.serve.ai_inference_recommender.result import (
    BenchmarkMetrics, PROFILE_EXPORT_FILENAME, OUTPUT_ARCHIVE_FILENAME,
)

def _node(key):
    m = re.search(r"/benchmark/([^/]+)/", key)
    if m: return m.group(1)
    parts = key.split("/")
    return parts[-4] if len(parts) >= 4 else "?"

def _conc(key):
    m = re.search(r"/c(\d+)/", key)
    return int(m.group(1)) if m else None

S3_OUT = getattr(getattr(rec, "output_config", None), "s3_output_location", None)
curve = []
if not S3_OUT:
    print("recommendation job exposes no s3_output_location; nothing to read.")
else:
    print("raw artifacts under:", S3_OUT)
    _bucket, _prefix = S3_OUT.split("/")[2], "/".join(S3_OUT.split("/")[3:])
    s3 = boto3.client("s3")
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=_bucket, Prefix=_prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if not key.endswith(OUTPUT_ARCHIVE_FILENAME):
                continue
            body = s3.get_object(Bucket=_bucket, Key=key)["Body"].read()
            with tarfile.open(fileobj=io.BytesIO(body), mode="r:gz") as tar:
                mem = next((t for t in tar.getmembers()
                            if t.name.endswith(PROFILE_EXPORT_FILENAME)), None)
                if mem is None:
                    continue
                profile = json.loads(tar.extractfile(mem).read().decode("utf-8"))
            m = BenchmarkMetrics.from_profile_json(profile)
            curve.append({"node": _node(key), "concurrency": _conc(key),
                          "tok_s": getattr(m.output_token_throughput, "avg", None),
                          "itl_p50": getattr(m.inter_token_latency, "p50", None),
                          "ttft_p50": getattr(m.time_to_first_token, "p50", None)})

if not curve:
    print("No per-concurrency artifacts found yet.")
else:
    print(f"\n{'node':26} {'conc':>5} {'tok/s':>8} {'ITL p50 ms':>11} {'TTFT p50 ms':>12}")
    for row in sorted(curve, key=lambda r: (r["node"], r["concurrency"] or 0)):
        conc = row["concurrency"] if row["concurrency"] is not None else "-"
        print(f"{row['node'][:26]:26} {conc:>5} {row['tok_s'] or 0:>8.0f}"
              f" {row['itl_p50'] or 0:>11.1f} {row['ttft_p50'] or 0:>12.1f}")

In [ ]:
# Matched-concurrency view: for each concurrency benchmarked on 2+ nodes, compare tok/s side by side.
by_conc = {}
for row in curve:
    if row["concurrency"] is not None:
        by_conc.setdefault(row["concurrency"], []).append(row)

print("matched-concurrency (tok/s by node):")
for conc in sorted(by_conc):
    grp = by_conc[conc]
    if len(grp) < 2:
        continue
    cells = "   ".join(f"{g['node'][:16]}: {g['tok_s'] or 0:.0f}" for g in grp)
    print(f"  c{conc:<4} {cells}")

## 4. (optional) Optimize path

`advanced_optimization=True` runs the optimization exploration; it needs a dataset workload (`Workload.from_dataset`), not synthetic kwargs.

In [ ]:
from sagemaker.serve import Workload

mb_opt = ModelBuilder(s3_model_data_url=MODEL_S3, role_arn=ROLE)
workload = Workload.from_dataset(
    s3_uri="s3://<YOUR_BUCKET>/traces/openai-chat.jsonl",   # your request trace (JSONL)
    custom_dataset_type="openai-chat",
    tokenizer="gpt2",
)
opt = mb_opt.generate_deployment_recommendations(
    workload,
    PerformanceTarget.THROUGHPUT,
    instance_types=G7_INSTANCES,
    advanced_optimization=True,
    framework=InferenceFramework.VLLM,
    wait=True,
)
print("status:", opt.ai_recommendation_job_status)
print(mb_opt.recommendations)

## Cleanup

In [ ]:
from sagemaker.core.resources import AIWorkloadConfig

def _try(l, f):
    try: f()
    except Exception as e: print(f"  [skip] {l}: {e}")

for j in [x for x in (globals().get("rec"), globals().get("opt")) if x is not None]:
    _try(f"rec job {j.get_name()}", j.delete)
    _try(f"workload cfg {j.ai_workload_config_identifier}",
         lambda jj=j: AIWorkloadConfig.get(ai_workload_config_name=jj.ai_workload_config_identifier).delete())